In [3]:
import numpy as np
import os
from scipy.io import savemat, loadmat
import warnings
warnings.filterwarnings("ignore")
from ES_MDA import ES_MDA
import pandas as pd
import utility
import forward_model
import sobol_seq
import copy
from write_script import write_script

In [4]:
# initialization
Num_ens=10
Dim_est=3
Na=4
Alpha=np.array([9.333,7.0,4.0,2.0])
#ncores=10

#load observation data
obs=np.loadtxt('observation.txt')

var_obs = np.ones_like(obs)
obs=np.array([obs])
obs=obs.T

# set the error for different kinds of measurements
var_obs[:]=0.01 # SP
R=np.diag(var_obs)

s=np.zeros((Num_ens,Dim_est,Na+1))
data = sobol_seq.i4_sobol_generate(dim_num= Dim_est, n = Num_ens)
# set upper bound for all parameters
para_u = np.array([-9, -3, 5])
# set lower bound for all parameters
para_l = np.array([-17, -6.5, 0.5])
data1 = para_l+data*(para_u-para_l)
s_temp = data1
s_temp[:, 0] = np.log((s_temp[:, 0] - para_l[0]) / (para_u[0] - s_temp[:, 0]))
s_temp[:, 1] = np.log((s_temp[:, 1] - para_l[1]) / (para_u[1] - s_temp[:, 1]))
s_temp[:, 2] = np.log((s_temp[:, 2] - para_l[2]) / (para_u[2] - s_temp[:, 2]))
#s_temp=loadmat('s_tem1.mat')['s_tem'] #useful for restarting the inversion if it crashed for some reasons, else comment out
s[:,:,0]=s_temp

In [5]:
s_tempp=copy.deepcopy(s_temp)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_temp[:, 0]) / (1 + np.exp(s_temp[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_temp[:, 1]) / (1 + np.exp(s_temp[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_temp[:, 2]) / (1 + np.exp(s_temp[:, 2])))
write=10**s_tempp

In [12]:
for i in range(Num_ens):
    write_script(write[i][0],write[i][1],i+1)
    np.savetxt('param'+str(i+1)+'.txt',[write[i][2]])

In [ ]:
t =0
sim_obs= # shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))

rmse=np.sqrt(np.mean((np.mean(sim_obs,axis=0)-obs.flatten())**2))
print('RMSE ite_', t, ' : ', rmse) # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_', t, ' : ', nse_mean)

# 写入 metrics.txt 文件（追加模式）
with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

s[:,:,t+1] = ES_MDA(Num_ens, s[:,:,t], obs, sim_obs, Alpha[t], R, [], 2)
s_tem=s[:,:,t+1]
savemat('./s_tem' + str(t+1) + '.mat', {'s_tem':s_tem}) # save s for each step

In [ ]:
s_tempp=copy.deepcopy(s_tem)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_tem[:, 0]) / (1 + np.exp(s_tem[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_tem[:, 1]) / (1 + np.exp(s_tem[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_tem[:, 2]) / (1 + np.exp(s_tem[:, 2])))
write=10**s_tempp

In [12]:
for i in range(Num_ens):
    write_script(write[i][0],write[i][1],i+1)
    np.savetxt('param'+str(i+1)+'.txt',[write[i][2]])

In [ ]:
t =1
sim_obs= # shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))

rmse=np.sqrt(np.mean((np.mean(sim_obs,axis=0)-obs.flatten())**2))
print('RMSE ite_', t, ' : ', rmse) # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_', t, ' : ', nse_mean)

# 写入 metrics.txt 文件（追加模式）
with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

s[:,:,t+1] = ES_MDA(Num_ens, s[:,:,t], obs, sim_obs, Alpha[t], R, [], 2)
s_tem=s[:,:,t+1]
savemat('./s_tem' + str(t+1) + '.mat', {'s_tem':s_tem}) # save s for each step

In [ ]:
s_tempp=copy.deepcopy(s_tem)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_tem[:, 0]) / (1 + np.exp(s_tem[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_tem[:, 1]) / (1 + np.exp(s_tem[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_tem[:, 2]) / (1 + np.exp(s_tem[:, 2])))
write=10**s_tempp

In [12]:
for i in range(Num_ens):
    write_script(write[i][0],write[i][1],i+1)
    np.savetxt('param'+str(i+1)+'.txt',[write[i][2]])

In [ ]:
t =2
sim_obs= # shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))

rmse=np.sqrt(np.mean((np.mean(sim_obs,axis=0)-obs.flatten())**2))
print('RMSE ite_', t, ' : ', rmse) # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_', t, ' : ', nse_mean)

# 写入 metrics.txt 文件（追加模式）
with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

s[:,:,t+1] = ES_MDA(Num_ens, s[:,:,t], obs, sim_obs, Alpha[t], R, [], 2)
s_tem=s[:,:,t+1]
savemat('./s_tem' + str(t+1) + '.mat', {'s_tem':s_tem}) # save s for each step

In [ ]:
s_tempp=copy.deepcopy(s_tem)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_tem[:, 0]) / (1 + np.exp(s_tem[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_tem[:, 1]) / (1 + np.exp(s_tem[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_tem[:, 2]) / (1 + np.exp(s_tem[:, 2])))
write=10**s_tempp

In [12]:
for i in range(Num_ens):
    write_script(write[i][0],write[i][1],i+1)
    np.savetxt('param'+str(i+1)+'.txt',[write[i][2]])

In [ ]:
t =3
sim_obs= # shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))

rmse=np.sqrt(np.mean((np.mean(sim_obs,axis=0)-obs.flatten())**2))
print('RMSE ite_', t, ' : ', rmse) # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_', t, ' : ', nse_mean)

# 写入 metrics.txt 文件（追加模式）
with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

s[:,:,t+1] = ES_MDA(Num_ens, s[:,:,t], obs, sim_obs, Alpha[t], R, [], 2)
s_tem=s[:,:,t+1]
savemat('./s_tem' + str(t+1) + '.mat', {'s_tem':s_tem}) # save s for each step

In [ ]:
s_tempp=copy.deepcopy(s_tem)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_tem[:, 0]) / (1 + np.exp(s_tem[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_tem[:, 1]) / (1 + np.exp(s_tem[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_tem[:, 2]) / (1 + np.exp(s_tem[:, 2])))
write=10**s_tempp

In [12]:
for i in range(Num_ens):
    write_script(write[i][0],write[i][1],i+1)
    np.savetxt('param'+str(i+1)+'.txt',[write[i][2]])

In [ ]:
sim_obs=model.run_model(s[:,:,len(Alpha)],t+1) # shape of sim_obs (Num_ens,Num_obs)
np.savetxt('./sim_obs' + str(len(Alpha)) + '.txt', np.mean(sim_obs,axis=0))

rmse = np.sqrt(np.mean((np.mean(sim_obs, axis=0) - obs.flatten()) ** 2))
print('RMSE ite_', t, ' : ', rmse)  # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_',  len(Alpha), ' : ', nse_mean)

with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t+1}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

savemat('results_esmda.mat',{'s':s,'sim_obs':sim_obs}) # sim_obs for the last step

In [11]:
cd ../param1
sbatch model.sh
cd ../param2
sbatch model.sh
cd ../param3
sbatch model.sh
cd ../param4
sbatch model.sh
cd ../param5
sbatch model.sh
cd ../param6
sbatch model.sh
cd ../param7
sbatch model.sh
cd ../param8
sbatch model.sh
cd ../param9
sbatch model.sh
cd ../param10
sbatch model.sh

SyntaxError: invalid syntax (374403859.py, line 1)

In [ ]:
cd ../param1
post_run.py
cd ../param2
post_run.py
cd ../param3
post_run.py
cd ../param4
post_run.py
cd ../param5
post_run.py
cd ../param6
post_run.py
cd ../param7
post_run.py
cd ../param8
post_run.py
cd ../param9
post_run.py
cd ../param10
post_run.py

In [ ]:
git add .
git commit -m 'finalize inversion'
git push origin short_model_hourly
